# 화재 이미지 탐지 — Kaggle GPU 학습

**설정 방법:**
1. 우측 Accelerator → **GPU T4 x2** 선택
2. 데이터셋 추가 (2개 모두):
   - `+ Add Data` → `yuntarwon/fireimage-detection` (abnormal 2,871장)
   - `+ Add Data` → `yuntarwon/fireimage-normal` (normal 6,106장)
3. 셀 순서대로 실행

In [ ]:
# Cell 1: 레포 클론
import os

!git clone https://github.com/yuntaewon812/fireimage_detection.git /kaggle/working/fireimage_detection
os.chdir('/kaggle/working/fireimage_detection')
print('작업 디렉토리:', os.getcwd())

In [ ]:
# Cell 2: 추가 패키지 설치 (torch/torchvision은 Kaggle에 기본 설치됨)
!pip install timm transformers einops grad-cam lime shap umap-learn -q

In [ ]:
# Cell 3: 두 데이터셋 연결
# - yuntarwon/fireimage-detection → data/fireimage/abnormal/
# - yuntarwon/fireimage-normal    → data/fireimage/normal/
import os, glob

base = '/kaggle/working/fireimage_detection/data/fireimage'
os.makedirs(base, exist_ok=True)

# abnormal 연결
abnormal_src = '/kaggle/input/fireimage-detection/abnormal'
abnormal_dst = f'{base}/abnormal'
assert os.path.exists(abnormal_src), f'fireimage-detection 데이터셋을 추가했는지 확인하세요: {abnormal_src}'
if not os.path.exists(abnormal_dst):
    os.symlink(abnormal_src, abnormal_dst)
print('abnormal 링크 완료:', abnormal_src)

# normal 연결 (zip 압축 해제 후 루트에 바로 이미지 있음)
normal_src = '/kaggle/input/fireimage-normal'
normal_dst = f'{base}/normal'
assert os.path.exists(normal_src), f'fireimage-normal 데이터셋을 추가했는지 확인하세요: {normal_src}'
if not os.path.exists(normal_dst):
    os.symlink(normal_src, normal_dst)
print('normal 링크 완료:', normal_src)

# 검증
normal_cnt   = sum(1 for f in glob.glob(f'{base}/normal/**/*',   recursive=True) if os.path.isfile(f))
abnormal_cnt = sum(1 for f in glob.glob(f'{base}/abnormal/**/*', recursive=True) if os.path.isfile(f))
print(f'normal: {normal_cnt:,}장 / abnormal: {abnormal_cnt:,}장')
assert normal_cnt > 0 and abnormal_cnt > 0, '이미지를 찾지 못했습니다.'

In [ ]:
# Cell 4: GPU + 데이터 확인
import torch

print('GPU 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

normal_cnt = sum(1 for _ in glob.glob('/kaggle/working/fireimage_detection/data/fireimage/normal/**/*', recursive=True) if os.path.isfile(_))
abnormal_cnt = sum(1 for _ in glob.glob('/kaggle/working/fireimage_detection/data/fireimage/abnormal/**/*', recursive=True) if os.path.isfile(_))
print(f'normal: {normal_cnt:,}장 / abnormal: {abnormal_cnt:,}장')

In [ ]:
# Cell 5: 학습 실행 (fold0~fold2, 7개 모델 전체 — 약 15~25 GPU 시간 예상)
# 주의: 세션이 끊기면 완료된 모델까지의 model_save를 반드시 다운로드해두세요
!python main_v2.py --class_name fireimage

In [ ]:
# Cell 6: 결과 확인 및 비교표 생성
!python compare_models.py

import pandas as pd
df = pd.read_csv('results/fireimage/metrics.csv')
print(df.to_string())

In [ ]:
# Cell 7: model_save 압축 (세션 종료 전 반드시 실행 → Output에서 다운로드)
import shutil
shutil.make_archive('/kaggle/working/model_save_backup', 'zip', '/kaggle/working/fireimage_detection', 'model_save')
shutil.make_archive('/kaggle/working/results_backup', 'zip', '/kaggle/working/fireimage_detection', 'results')
print('압축 완료. Output 탭에서 다운로드하세요.')